# Math 5422 — Project 2: Delta-Hedged Option (QuantLib Version)

This is the **QuantLib-based** implementation of Project 2:

- Long a **2M 110% OTM Call** on **TSLA** starting **2021-10-18**, expiring **2021-12-17**
- Maintain a **delta-neutral hedge** (rebalanced daily)
- Compute **daily P&L** and **P&L attribution** to: Delta, Gamma, Theta, Vega, Rho
- Volatility is obtained by **linear interpolation in moneyness** (and optionally interpolated between 1M and 2M tenors)

## Important
- This notebook uses `QuantLib` (Python bindings).  
- If `import QuantLib as ql` fails, install it and rerun the notebook.
  - Common install: `pip install QuantLib-Python`

(QuantLib note: option greeks are computed by the pricing engine; theta is typically *annualized* in QuantLib—if you use `option.theta()` directly, divide by 365 to convert to per-day. This notebook’s **theta P&L** uses the project’s repricing formula, so it does **not** rely on that convention.)


In [1]:
import math
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("Project 2 Tesla Data.xlsx")
SAMPLE_PATH = Path("Sample PnL Output Table.xlsx")

try:
    import QuantLib as ql
    QL_OK = True
except Exception as e:
    QL_OK = False
    QL_IMPORT_ERROR = e

QL_OK, (None if QL_OK else str(QL_IMPORT_ERROR))


(True, None)

In [2]:
# -----------------------------
# 1) Load and clean input data
# -----------------------------
swap_raw = pd.read_excel(DATA_PATH, sheet_name="Swap Curve", header=None)

tickers = swap_raw.loc[3, 1:14].tolist()
swap = swap_raw.loc[5:, 0:14].copy()
swap.columns = ["Date"] + tickers
swap["Date"] = pd.to_datetime(swap["Date"])
swap = swap.dropna(subset=["Date"]).sort_values("Date")

sv_raw = pd.read_excel(DATA_PATH, sheet_name="Stock and Vol Data", header=None)

m_labels = sv_raw.loc[1, 2:8].tolist()
m_levels = [1.3, 1.2, 1.1, 1.0, 0.9, 0.8, 0.7]  # nodes interpreted as K/S
label_map = dict(zip(m_labels, m_levels))

sv = sv_raw.loc[3:, :].copy()
sv.columns = (
    ["Date", "Spot"]
    + [f"1M_{label_map[x]:.1f}" for x in m_labels]
    + [f"2M_{label_map[x]:.1f}" for x in m_labels]
)

sv["Date"] = pd.to_datetime(sv["Date"])
sv["Spot"] = pd.to_numeric(sv["Spot"], errors="coerce")
sv = sv.dropna(subset=["Date", "Spot"]).sort_values("Date")

for c in sv.columns:
    if c.startswith(("1M_", "2M_")):
        sv[c] = pd.to_numeric(sv[c], errors="coerce") / 100.0  # percent -> decimal

sv = sv.groupby("Date", as_index=False).last()

swap.head(), sv.head()


(        Date US0003M Index US0012M Index USSW2 Curncy USSW3 Curncy  \
 5 2021-10-01       0.13313       0.23488       0.3713       0.6289   
 6 2021-10-04       0.12663         0.232       0.3733       0.6336   
 7 2021-10-05         0.124       0.23688       0.3864       0.6544   
 8 2021-10-06         0.124       0.24113       0.3949       0.6649   
 9 2021-10-07       0.12363       0.24313       0.4125       0.6978   
 
   USSW4 Curncy USSW5 Curncy USSW6 Curncy USSW7 Curncy USSW8 Curncy  \
 5       0.8476       1.0187       1.1561       1.2662       1.3515   
 6       0.8522       1.0245       1.1622       1.2736       1.3605   
 7       0.8766        1.051       1.1961        1.311       1.4007   
 8       0.8885       1.0619       1.2032       1.3155       1.4027   
 9       0.9276       1.1063       1.2511       1.3687       1.4587   
 
   USSW9 Curncy USSW10 Curncy USSW15 Curncy USSW20 Curncy USSW30 Curncy  
 5       1.4204        1.4798        1.6662         1.747        1.768

In [3]:
# -----------------------------
# 2) Vol interpolation helpers
# -----------------------------
MONEyness_NODES = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3])

def interp_smile(row, prefix: str, m: float) -> float:
    ys = np.array([row[f"{prefix}_{x:.1f}"] for x in MONEyness_NODES], dtype=float)
    return float(np.interp(m, MONEyness_NODES, ys))

def interp_tenor_vol(row, m: float, tenor_months: float) -> float:
    sig1 = interp_smile(row, "1M", m)
    sig2 = interp_smile(row, "2M", m)
    w = float(np.clip(tenor_months, 1.0, 2.0) - 1.0)  # 0 at 1M, 1 at 2M
    return (1.0 - w) * sig1 + w * sig2


In [4]:
# -----------------------------
# 3) QuantLib pricer + greeks
# -----------------------------
def to_ql_date(ts: pd.Timestamp) -> "ql.Date":
    return ql.Date(ts.day, ts.month, ts.year)

def ql_european_call_price_greeks(
    S: float,
    K: float,
    r: float,
    q: float,
    sigma: float,
    valuation_date: pd.Timestamp,
    expiry_date: pd.Timestamp
):
    """Return (price, delta, gamma, vega, rho, theta) using QuantLib.

    Conventions:
      - Rates and vol are decimals (e.g., 0.05 = 5%)
      - Greeks returned are QuantLib's native conventions.
        - `theta` is commonly annualized in QuantLib (divide by 365 for per-day if needed).
      - We use flat term structures and constant vol (as requested).
    """
    if not QL_OK:
        raise RuntimeError(f"QuantLib import failed: {QL_IMPORT_ERROR}")

    day_count = ql.Actual365Fixed()
    calendar = ql.NullCalendar()

    eval_date = to_ql_date(valuation_date)
    exp_date = to_ql_date(expiry_date)

    ql.Settings.instance().evaluationDate = eval_date

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(eval_date, r, day_count))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(eval_date, q, day_count))
    vol_ts = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(eval_date, calendar, sigma, day_count))

    process = ql.BlackScholesMertonProcess(spot, q_ts, r_ts, vol_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    exercise = ql.EuropeanExercise(exp_date)
    option = ql.VanillaOption(payoff, exercise)
    option.setPricingEngine(ql.AnalyticEuropeanEngine(process))

    price = option.NPV()
    delta = option.delta()
    gamma = option.gamma()
    vega  = option.vega()
    rho   = option.rho()
    theta = option.theta()

    return float(price), float(delta), float(gamma), float(vega), float(rho), float(theta)


In [5]:
# -----------------------------
# 4) Build daily backtest table
# -----------------------------
START_DATE = pd.Timestamp("2021-10-18")
EXPIRY_DATE = pd.Timestamp("2021-12-17")

OPT_QTY = 1000
DIV_YIELD = 0.0

rates = swap[["Date", "US0003M Index"]].copy()
rates["US0003M Index"] = pd.to_numeric(rates["US0003M Index"], errors="coerce")
rates = rates.sort_values("Date").ffill()

df = sv[(sv["Date"] >= START_DATE) & (sv["Date"] <= EXPIRY_DATE)].copy()
df = df.merge(rates, on="Date", how="left")
df["US0003M Index"] = df["US0003M Index"].astype(float).ffill()
df["r"] = df["US0003M Index"] / 100.0

S0 = float(df.iloc[0]["Spot"])
K = 1.10 * S0

df["tau"] = (EXPIRY_DATE - df["Date"]).dt.days / 365.0
df["tenor_m"] = df["tau"] * 12.0
df["m"] = K / df["Spot"]

df["IV"] = df.apply(lambda r: interp_tenor_vol(r, float(r["m"]), float(r["tenor_m"])), axis=1)

# Price + Greeks via QuantLib for each day
rows = []
for _, r in df.iterrows():
    price, delta, gamma, vega, rho, theta = ql_european_call_price_greeks(
        S=float(r["Spot"]),
        K=K,
        r=float(r["r"]),
        q=DIV_YIELD,
        sigma=float(r["IV"]),
        valuation_date=r["Date"],
        expiry_date=EXPIRY_DATE
    )
    rows.append((price, delta, gamma, vega, rho, theta))

tmp = pd.DataFrame(rows, columns=["OptionPrice","Delta","Gamma","Vega","Rho","Theta"])
df = pd.concat([df.reset_index(drop=True), tmp], axis=1)

df["StockQuantity"] = (-OPT_QTY * df["Delta"]).round().astype(int)

df[["Date","Spot","IV","r","OptionPrice","Delta","StockQuantity"]].head()


,Date,Spot,IV,r,OptionPrice,Delta,StockQuantity
0,2021-10-18,870.11,0.344830,0.001315,18.821678,0.270842,-271
1,2021-10-19,864.27,0.349271,0.001295,17.455709,0.256239,-256
2,2021-10-20,865.80,0.345809,0.001283,17.129411,0.255581,-256
3,2021-10-21,894.00,0.357263,0.001239,26.533924,0.340436,-340
4,2021-10-22,909.68,0.362980,0.001249,32.578612,0.387775,-388


In [6]:
# -----------------------------
# 5) Daily P&L + attribution (repricing formulas)
# -----------------------------
def ql_price_only(S, K, r, q, sigma, valuation_date, expiry_date):
    return ql_european_call_price_greeks(S, K, r, q, sigma, valuation_date, expiry_date)[0]

records = []
for i in range(len(df)):
    row = df.iloc[i]
    records.append({
        "Dates": row["Date"],
        "OptionQuantity": OPT_QTY,
        "OptionPrice": float(row["OptionPrice"]),
        "StockQuantity": int(row["StockQuantity"]),
        "StockPrice": float(row["Spot"]),
        "IV": float(row["IV"]),
        "Delta": float(row["Delta"]),
        "DeltaPNL": np.nan,
        "GammaPNL": np.nan,
        "ThetaPNL": np.nan,
        "VegaPNL": np.nan,
        "RhoPNL": np.nan,
        "OptionPNL": np.nan,
        "StockPNL": np.nan,
        "TotalPNL": np.nan,
    })

out = pd.DataFrame(records)

for i in range(1, len(df)):
    prev = df.iloc[i-1]
    cur  = df.iloc[i]

    S1, S2 = float(prev["Spot"]), float(cur["Spot"])
    sig1, sig2 = float(prev["IV"]), float(cur["IV"])
    r1, r2 = float(prev["r"]), float(cur["r"])
    T1_date, T2_date = prev["Date"], cur["Date"]

    # Base prices
    V1 = ql_price_only(S1, K, r1, DIV_YIELD, sig1, T1_date, EXPIRY_DATE)
    V2 = ql_price_only(S2, K, r2, DIV_YIELD, sig2, T2_date, EXPIRY_DATE)

    # Repricing ladder per project spec (keep expiry fixed, move evaluation date/inputs)
    # Theta: V(S2,σ2,r2,T2) - V(S2,σ2,r2,T1)
    V_S2_sig2_r2_T1 = ql_price_only(S2, K, r2, DIV_YIELD, sig2, T1_date, EXPIRY_DATE)

    # Rho: V(S2,σ2,r2,T1) - V(S2,σ2,r1,T1)
    V_S2_sig2_r1_T1 = ql_price_only(S2, K, r1, DIV_YIELD, sig2, T1_date, EXPIRY_DATE)

    # Vega: V(S2,σ2,r1,T1) - V(S2,σ1,r1,T1)
    V_S2_sig1_r1_T1 = ql_price_only(S2, K, r1, DIV_YIELD, sig1, T1_date, EXPIRY_DATE)

    theta_pnl = (V2 - V_S2_sig2_r2_T1) * OPT_QTY
    rho_pnl   = (V_S2_sig2_r2_T1 - V_S2_sig2_r1_T1) * OPT_QTY
    vega_pnl  = (V_S2_sig2_r1_T1 - V_S2_sig1_r1_T1) * OPT_QTY

    # Delta + Gamma: V(S2,σ1,r1,T1) - V(S1,σ1,r1,T1)
    dg_pnl = (V_S2_sig1_r1_T1 - V1) * OPT_QTY

    delta_prev = float(prev["Delta"])
    dS = S2 - S1
    delta_pnl = (delta_prev * dS) * OPT_QTY
    gamma_pnl = dg_pnl - delta_pnl

    option_pnl = (V2 - V1) * OPT_QTY

    stock_qty_prev = int(out.loc[i-1, "StockQuantity"])
    stock_pnl = stock_qty_prev * (S2 - S1)

    total_pnl = option_pnl + stock_pnl

    out.loc[i, ["DeltaPNL","GammaPNL","ThetaPNL","VegaPNL","RhoPNL","OptionPNL","StockPNL","TotalPNL"]] = [
        delta_pnl, gamma_pnl, theta_pnl, vega_pnl, rho_pnl, option_pnl, stock_pnl, total_pnl
    ]

out.head()


,Dates,OptionQuantity,OptionPrice,StockQuantity,StockPrice,IV,Delta,DeltaPNL,GammaPNL,ThetaPNL,VegaPNL,RhoPNL,OptionPNL,StockPNL,TotalPNL
0,2021-10-18,1000,18.821678,-271,870.11,0.344830,0.270842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-10-19,1000,17.455709,-256,864.27,0.349271,0.256239,-1581.719246,46.065268,-331.160538,501.521034,-0.675748,-1365.969230,1582.64,216.670770
2,2021-10-20,1000,17.129411,-256,865.80,0.345809,0.255581,392.045563,3.111338,-330.817590,-390.221023,-0.415665,-326.297376,-391.68,-717.977376
3,2021-10-21,1000,26.533924,-340,894.00,0.357263,0.340436,7207.377780,1110.634626,-405.454950,1493.892765,-1.937380,9404.512842,-7219.20,2185.312842
4,2021-10-22,1000,32.578612,-388,909.68,0.362980,0.387775,5338.036365,360.593852,-441.718908,787.275589,0.501368,6044.688267,-5331.20,713.488267


In [ ]:
# -----------------------------
# 6) Match sample output style + save
# -----------------------------
def ordinal(n: int) -> str:
    if 10 <= n % 100 <= 20:
        suf = "th"
    else:
        suf = {1:"st",2:"nd",3:"rd"}.get(n % 10, "th")
    return f"{n}{suf}"

def format_date(dt: pd.Timestamp) -> str:
    return f"{dt.strftime('%B')} {ordinal(dt.day)}, {dt.year}"

final = out.copy()
final["Dates"] = final["Dates"].apply(format_date)

final = final[
    ["Dates","OptionQuantity","OptionPrice","StockQuantity","StockPrice","IV","Delta",
     "DeltaPNL","GammaPNL","ThetaPNL","VegaPNL","RhoPNL",
     "OptionPNL","StockPNL","TotalPNL"]
]

final.to_excel("Project2_QuantLib_DeltaHedge_PnL_Output.xlsx", index=False)

final.head()


,Dates,OptionQuantity,OptionPrice,StockQuantity,StockPrice,IV,Delta,DeltaPNL,GammaPNL,ThetaPNL,VegaPNL,RhoPNL,OptionPNL,StockPNL,TotalPNL
0,"October 18th, 2021",1000,18.821678,-271,870.11,0.344830,0.270842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"October 19th, 2021",1000,17.455709,-256,864.27,0.349271,0.256239,-1581.719246,46.065268,-331.160538,501.521034,-0.675748,-1365.969230,1582.64,216.670770
2,"October 20th, 2021",1000,17.129411,-256,865.80,0.345809,0.255581,392.045563,3.111338,-330.817590,-390.221023,-0.415665,-326.297376,-391.68,-717.977376
3,"October 21st, 2021",1000,26.533924,-340,894.00,0.357263,0.340436,7207.377780,1110.634626,-405.454950,1493.892765,-1.937380,9404.512842,-7219.20,2185.312842
4,"October 22nd, 2021",1000,32.578612,-388,909.68,0.362980,0.387775,5338.036365,360.593852,-441.718908,787.275589,0.501368,6044.688267,-5331.20,713.488267
